# News Article Engagement Analysis: Digital Media Platform
**Analyst:** Zari Syed | **Date:** May 2025 | **Domain:** Editorial analytics & content strategy

---

## Overview

Understanding what drives reader engagement is critical for a digital news publisher.
This analysis explores content performance across sections, publication timing, article length, and topics.

> **Data note:** This notebook uses realistically simulated article data modelled on patterns typical
> of quality UK news publishers. The code cell below shows exactly how to replace this with live
> Guardian API data using their free developer key.

**Key questions:**
1. Which sections generate the most content and highest engagement?
2. When do readers engage most -- which days and hours perform best?
3. Does article length correlate with engagement?
4. Which topics are trending and what does that mean for editorial strategy?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
np.random.seed(7)
print("Libraries loaded.")

## 1. Data Source

The Guardian provides a free public API for article metadata. The code below shows how to fetch real data — replace the synthetic generation section with this to run on live articles.

```python
# LIVE DATA FETCH (requires free API key from open-platform.theguardian.com)
# import requests
# def fetch_guardian(section, pages=5, api_key='your-key-here'):
#     results = []
#     for page in range(1, pages+1):
#         r = requests.get('https://content.guardianapis.com/search', params={
#             'section': section, 'show-fields': 'wordcount',
#             'show-tags': 'keyword', 'page-size': 50,
#             'page': page, 'api-key': api_key
#         })
#         results.extend(r.json()['response']['results'])
#     return results
```

For this analysis, we generate data that mirrors the distribution patterns of real article metadata.

In [ ]:
SECTIONS = {
    'news':        {'weight': 0.28, 'avg_words': 850,  'std_words': 350, 'base_eng': 72},
    'politics':    {'weight': 0.18, 'avg_words': 950,  'std_words': 400, 'base_eng': 85},
    'technology':  {'weight': 0.14, 'avg_words': 780,  'std_words': 280, 'base_eng': 91},
    'culture':     {'weight': 0.10, 'avg_words': 720,  'std_words': 250, 'base_eng': 68},
    'sport':       {'weight': 0.12, 'avg_words': 600,  'std_words': 200, 'base_eng': 78},
    'business':    {'weight': 0.10, 'avg_words': 900,  'std_words': 350, 'base_eng': 62},
    'environment': {'weight': 0.08, 'avg_words': 820,  'std_words': 300, 'base_eng': 74},
}
KEYWORDS = {
    'news':        ['ukraine','inflation','nhs','election','migration','housing'],
    'politics':    ['labour','conservatives','budget','reform','keir starmer','policy'],
    'technology':  ['ai','openai','google','meta','apple','cybersecurity'],
    'culture':     ['film','television','music','books','theatre','streaming'],
    'sport':       ['football','premier league','cricket','tennis','olympics','transfer'],
    'business':    ['economy','interest rates','ftse','investment','startups','trade'],
    'environment': ['climate','renewable','emissions','biodiversity','net zero','flooding'],
}
hour_probs = np.array([1,1,1,1,1,2,3,8,9,6,5,5,5,5,8,9,6,5,4,4,3,2,2,1], dtype=float)
hour_probs /= hour_probs.sum()

rows = []
start_date = datetime(2025, 1, 1)
for _ in range(2400):
    section = np.random.choice(list(SECTIONS.keys()), p=[v['weight'] for v in SECTIONS.values()])
    s = SECTIONS[section]
    hour = int(np.random.choice(24, p=hour_probs))
    pub_date = start_date + timedelta(days=int(np.random.randint(0,89)),
                                      hours=hour, minutes=int(np.random.randint(0,60)))
    wordcount = max(100, int(np.random.normal(s['avg_words'], s['std_words'])))
    time_bonus = 8 if 7 <= hour <= 9 else (5 if 14 <= hour <= 16 else 0)
    engagement = float(np.clip(
        s['base_eng'] * min(1.2, wordcount/s['avg_words']) + time_bonus + np.random.normal(0,12), 0, 100))
    rows.append({
        'section': section, 'pub_date': pub_date, 'weekday': pub_date.strftime('%A'),
        'hour': hour, 'wordcount': wordcount,
        'engagement_score': round(engagement, 1),
        'primary_keyword': np.random.choice(KEYWORDS[section]),
        'month': pub_date.strftime('%b %Y')
    })

df = pd.DataFrame(rows).sort_values('pub_date').reset_index(drop=True)
print("Dataset: {:,} articles | {} to {}".format(
    len(df), df['pub_date'].min().date(), df['pub_date'].max().date()))
print(df['section'].value_counts().to_string())

## 2. Content Volume by Section

Which sections produce the most content? Volume alone doesn't indicate impact, but it reveals editorial priorities and lets us compare productivity against engagement.

In [ ]:
section_stats = df.groupby('section').agg(
    articles=('section','count'),
    avg_engagement=('engagement_score','mean'),
    avg_wordcount=('wordcount','mean')
).round(1).sort_values('articles', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colours = sns.color_palette('muted', len(section_stats))

ax = axes[0]
bars = ax.barh(section_stats.index, section_stats['articles'], color=colours, alpha=0.85)
ax.set_xlabel('Number of Articles Published')
ax.set_title('Content Volume by Section (Jan-Mar 2025)')
for bar, val in zip(bars, section_stats['articles']):
    ax.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=9)
ax.invert_yaxis()

ax2 = axes[1]
for i, (sect, row) in enumerate(section_stats.iterrows()):
    ax2.scatter(row['articles'], row['avg_engagement'], s=row['avg_wordcount']/5,
                color=colours[i], alpha=0.8, label=sect, edgecolors='white', linewidth=0.5)
ax2.set_xlabel('Number of Articles')
ax2.set_ylabel('Average Engagement Score')
ax2.set_title('Engagement vs Volume (bubble = avg word count)')
ax2.legend(fontsize=8, loc='lower right')
plt.suptitle('Editorial Performance Overview', fontsize=14)
plt.tight_layout()
plt.show()
print(section_stats.to_string())

## 3. Publication Timing Analysis

When are articles published, and does timing affect engagement?
We look at hourly patterns, weekday patterns, and an engagement heatmap by day x hour.

In [ ]:
WEEKDAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
hourly = df.groupby('hour')['engagement_score'].agg(['count','mean']).reset_index()
weekday_counts = df.groupby('weekday')['engagement_score'].agg(['count','mean']).reindex(WEEKDAY_ORDER)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

ax = axes[0,0]
ax.bar(hourly['hour'], hourly['count'], color='#5B9BD5', alpha=0.75)
ax2 = ax.twinx()
ax2.plot(hourly['hour'], hourly['mean'], 'o-', color='#ED7D31', lw=2, ms=5)
ax2.set_ylabel('Avg Engagement', color='#ED7D31')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Articles Published')
ax.set_title('Volume & Engagement by Hour')

ax = axes[0,1]
ax.bar(range(7), weekday_counts['count'], color=['#5B9BD5']*5+['#C0504D']*2, alpha=0.8)
ax2b = ax.twinx()
ax2b.plot(range(7), weekday_counts['mean'], 'D--', color='#70AD47', lw=2, ms=6)
ax2b.set_ylabel('Avg Engagement', color='#70AD47')
ax.set_xticks(range(7))
ax.set_xticklabels([d[:3] for d in WEEKDAY_ORDER])
ax.set_ylabel('Articles Published')
ax.set_title('Volume & Engagement by Weekday')

ax = axes[1,0]
pivot = df.pivot_table(values='engagement_score', index='weekday', columns='hour', aggfunc='mean')
sns.heatmap(pivot.reindex(WEEKDAY_ORDER), cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Avg Engagement'}, linewidths=0.3)
ax.set_title('Engagement Heatmap: Day x Hour')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')

ax = axes[1,1]
monthly = df.groupby('month').agg(articles=('section','count'),
                                   engagement=('engagement_score','mean'))
monthly = monthly.reindex(['Jan 2025','Feb 2025','Mar 2025'])
ax.bar(range(3), monthly['articles'], color='#4BACC6', alpha=0.8)
ax2c = ax.twinx()
ax2c.plot(range(3), monthly['engagement'], 'o-', color='#F79646', lw=2.5, ms=8)
ax2c.set_ylabel('Avg Engagement', color='#F79646')
ax.set_xticks(range(3))
ax.set_xticklabels(monthly.index)
ax.set_ylabel('Articles Published')
ax.set_title('Monthly Volume & Engagement Trend')

plt.suptitle('Publication Timing Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Article Length & Engagement

Does word count affect how readers engage? We bucket articles by length and test whether longer articles consistently outperform shorter ones.

In [ ]:
bins = [0, 300, 600, 900, 1200, 1600, 9999]
labels = ['<300','300-600','600-900','900-1200','1200-1600','>1600']
df['length_bucket'] = pd.cut(df['wordcount'], bins=bins, labels=labels)
bucket_eng = df.groupby('length_bucket', observed=True)['engagement_score'].agg(['mean','sem','count'])
corr = df['wordcount'].corr(df['engagement_score'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.bar(range(len(labels)), bucket_eng['mean'],
       yerr=bucket_eng['sem']*1.96,
       color=sns.color_palette('Blues_d', len(labels)), alpha=0.85, capsize=5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=20)
ax.set_xlabel('Word Count Range')
ax.set_ylabel('Average Engagement Score')
ax.set_title('Engagement by Article Length (Error bars = 95% CI)')
for i, (_, row) in enumerate(bucket_eng.iterrows()):
    ax.text(i, row['mean']+0.5, 'n={}'.format(int(row['count'])), ha='center', fontsize=8)

ax2 = axes[1]
sect_list = list(df['section'].unique())
palette = sns.color_palette('muted', len(sect_list))
for i, section in enumerate(sect_list):
    sub = df[df['section']==section].sample(min(80, len(df[df['section']==section])))
    ax2.scatter(sub['wordcount'], sub['engagement_score'],
                color=palette[i], alpha=0.5, s=20, label=section)
z = np.polyfit(df['wordcount'], df['engagement_score'], 1)
xline = np.linspace(df['wordcount'].min(), df['wordcount'].max(), 100)
ax2.plot(xline, np.poly1d(z)(xline), 'k--', lw=1.5, alpha=0.6, label='Trend')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Engagement Score')
ax2.set_title('Word Count vs Engagement by Section  (r={:.3f})'.format(corr))
ax2.legend(fontsize=7, ncol=2)
plt.suptitle('Article Length and Engagement', fontsize=14)
plt.tight_layout()
plt.show()
print("Correlation: r = {:.3f}".format(corr))

## 5. Trending Topics

In [ ]:
top_keywords = df['primary_keyword'].value_counts().head(15)
kw_eng = df.groupby('primary_keyword')['engagement_score'].mean().sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

ax = axes[0]
bars = ax.barh(top_keywords.index[::-1], top_keywords.values[::-1], color='#5B9BD5', alpha=0.8)
ax.set_xlabel('Article Count')
ax.set_title('Top 15 Topics by Article Volume')
for bar, val in zip(bars, top_keywords.values[::-1]):
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=9)

ax2 = axes[1]
colours_kw = ['#70AD47' if v >= kw_eng.median() else '#C0504D' for v in kw_eng.values]
ax2.barh(kw_eng.index[::-1], kw_eng.values[::-1], color=colours_kw[::-1], alpha=0.8)
ax2.axvline(kw_eng.mean(), color='grey', ls='--', lw=1.2,
            label='Average ({:.1f})'.format(kw_eng.mean()))
ax2.set_xlabel('Average Engagement Score')
ax2.set_title('Top Topics by Engagement (green = above average)')
ax2.legend(fontsize=9)
plt.suptitle('Content Topics Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Key Findings & Editorial Recommendations

In [ ]:
best_sect = section_stats['avg_engagement'].idxmax()
hourly = df.groupby('hour')['engagement_score'].agg(['count','mean']).reset_index()
best_hour = int(hourly.loc[hourly['mean'].idxmax(), 'hour'])
weekday_counts = df.groupby('weekday')['engagement_score'].agg(['count','mean']).reindex(WEEKDAY_ORDER)
best_day = weekday_counts['mean'].idxmax()
best_bucket = bucket_eng['mean'].idxmax()

print("=" * 65)
print("KEY FINDINGS")
print("=" * 65)
print("")
print("1. SECTION PERFORMANCE")
print("   - Highest engagement: {} ({:.1f}/100 avg score)".format(
    best_sect, section_stats.loc[best_sect,'avg_engagement']))
print("   - Most articles published: {}".format(section_stats['articles'].idxmax()))
print("")
print("2. TIMING")
print("   - Best hour to publish: {}:00".format(best_hour))
print("   - Best day to publish : {}".format(best_day))
wknd = (weekday_counts.loc['Saturday','mean'] + weekday_counts.loc['Sunday','mean']) / 2
wkdy = weekday_counts.reindex(WEEKDAY_ORDER[:5])['mean'].mean()
print("   - Weekend vs weekday avg engagement: {:.1f} vs {:.1f}".format(wknd, wkdy))
print("")
print("3. ARTICLE LENGTH")
print("   - Sweet spot: {} words (highest avg engagement)".format(best_bucket))
print("   - Pearson r (word count vs engagement): {:.3f}".format(corr))
print("")
print("4. TOPICS")
print("   - Top by volume    : '{}' ({} articles)".format(top_keywords.index[0], top_keywords.iloc[0]))
print("   - Top by engagement: '{}' ({:.1f}/100)".format(kw_eng.index[0], kw_eng.iloc[0]))
print("")
print("RECOMMENDATIONS")
print("  1. Publish high-value content at {}:00-{}:00 for maximum engagement.".format(
    best_hour, best_hour+2))
print("  2. Prioritise {} for scheduling important articles.".format(best_day))
print("  3. Target {} words -- this range shows the strongest engagement.".format(best_bucket))
print("  4. Invest more in '{}' content -- highest engagement per article.".format(kw_eng.index[0]))
print("  5. Weekend content strategy needs review -- different topics or format may be needed.")